# Temperature Comparison: Panel vs Open-Meteo

This notebook visualizes:
1. Panel temperature (original, before weather adjustment)
2. Open-Meteo temperature
3. The subtraction (panel temperature - Open-Meteo baseline)

For both:
- All time
- Last two weeks

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from spotanomaly2.domain.weather_fetcher import WeatherFetcher

# Configuration
DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
PANEL_ID = '1'  # Change to '2' for panel 2
WEATHER_LATITUDE = 51.5
WEATHER_LONGITUDE = 10.5
WEATHER_LOOKBACK_DAYS = 14  # Last 2 weeks

print('Environment configured successfully!')

## Load Data

In [ ]:
# Load processed data (contains adjusted temperature)
df_processed = pd.read_parquet(PROCESSED_DIR / f'panel_{PANEL_ID}.parquet')
if df_processed.index.name != 'timestamp':
    if 'timestamp' in df_processed.columns:
        df_processed = df_processed.set_index('timestamp')
df_processed.index = pd.to_datetime(df_processed.index)

print(f'Loaded processed data: {df_processed.shape}')
print(f'Date range: {df_processed.index.min()} to {df_processed.index.max()}')
print(f'Columns: {list(df_processed.columns)}')

# Check if temperature column exists
if 'temperature' not in df_processed.columns:
    raise ValueError("Temperature column not found in processed data")

## Fetch Weather Data and Calculate Original Temperature

In [ ]:
# Initialize WeatherFetcher
weather_config = {
    "latitude": WEATHER_LATITUDE,
    "longitude": WEATHER_LONGITUDE,
    "use_forecast": True,
}
weather_fetcher = WeatherFetcher(weather_config)

# Fetch weather data for the entire date range
start_date = df_processed.index.min()
end_date = df_processed.index.max()

print(f'\nFetching weather data: {start_date} to {end_date}')

try:
    # Fetch weather data
    weather_df = weather_fetcher.get_weather_data(
        start=start_date,
        end=end_date,
        timezone="UTC",
        fallback_on_failure=False,
    )
    
    if "temperature_2m" not in weather_df.columns:
        raise ValueError("Weather data does not contain temperature_2m column")
    
    print(f'Successfully fetched {len(weather_df)} weather records')
    
    # Resample weather data to match panel data frequency (5min)
    panel_freq = "5min"
    weather_resampled = weather_df["temperature_2m"].resample(panel_freq).ffill()
    
    # Align weather data to panel data index
    weather_aligned = weather_resampled.reindex(df_processed.index, method="ffill")
    
    # Calculate rolling weather baseline (14-day rolling average)
    weather_baseline = weather_aligned.rolling(
        window=f"{WEATHER_LOOKBACK_DAYS}D",
        min_periods=1
    ).mean()
    
    # Fill any remaining NaN values with overall mean
    if weather_baseline.isna().any():
        overall_mean = weather_aligned.mean()
        weather_baseline = weather_baseline.fillna(overall_mean)
    
    # Calculate original temperature (adjusted temperature + weather baseline)
    # The processed data has: adjusted_temp = original_temp - weather_baseline
    # So: original_temp = adjusted_temp + weather_baseline
    adjusted_temperature = df_processed["temperature"]
    original_temperature = adjusted_temperature + weather_baseline
    
    # The subtraction is the adjusted temperature (original - Open-Meteo baseline)
    subtraction = adjusted_temperature
    
    # Open-Meteo temperature (the baseline used for adjustment)
    open_meteo_temp = weather_baseline
    
    print('Weather data processing completed successfully!')
    
except Exception as e:
    print(f'Error fetching weather data: {e}')
    raise

## Create Plots

In [ ]:
# Prepare data
timestamps = df_processed.index

# Calculate last two weeks cutoff
last_two_weeks_cutoff = timestamps.max() - pd.Timedelta(days=14)
mask_last_two_weeks = timestamps >= last_two_weeks_cutoff

# Create subplots: 3 rows (temperature, Open-Meteo, subtraction) x 2 columns (all time, last 2 weeks)
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Temperature - All Time',
        'Temperature - Last Two Weeks',
        'Open-Meteo - All Time',
        'Open-Meteo - Last Two Weeks',
        'Subtraction (Temperature - Open-Meteo) - All Time',
        'Subtraction (Temperature - Open-Meteo) - Last Two Weeks'
    ),
    vertical_spacing=0.08,
    horizontal_spacing=0.1
)

# Row 1: Temperature
# All time
fig.add_trace(
    go.Scatter(
        x=timestamps,
        y=original_temperature,
        mode='lines',
        name='Temperature',
        line=dict(color='blue', width=1),
        hovertemplate='<b>Temperature</b><br>Time: %{x}<br>Value: %{y:.2f}°C<extra></extra>'
    ),
    row=1, col=1
)

# Last two weeks
fig.add_trace(
    go.Scatter(
        x=timestamps[mask_last_two_weeks],
        y=original_temperature[mask_last_two_weeks],
        mode='lines',
        name='Temperature',
        line=dict(color='blue', width=1),
        showlegend=False,
        hovertemplate='<b>Temperature</b><br>Time: %{x}<br>Value: %{y:.2f}°C<extra></extra>'
    ),
    row=1, col=2
)

# Row 2: Open-Meteo
# All time
fig.add_trace(
    go.Scatter(
        x=timestamps,
        y=open_meteo_temp,
        mode='lines',
        name='Open-Meteo',
        line=dict(color='green', width=1),
        hovertemplate='<b>Open-Meteo</b><br>Time: %{x}<br>Value: %{y:.2f}°C<extra></extra>'
    ),
    row=2, col=1
)

# Last two weeks
fig.add_trace(
    go.Scatter(
        x=timestamps[mask_last_two_weeks],
        y=open_meteo_temp[mask_last_two_weeks],
        mode='lines',
        name='Open-Meteo',
        line=dict(color='green', width=1),
        showlegend=False,
        hovertemplate='<b>Open-Meteo</b><br>Time: %{x}<br>Value: %{y:.2f}°C<extra></extra>'
    ),
    row=2, col=2
)

# Row 3: Subtraction
# All time
fig.add_trace(
    go.Scatter(
        x=timestamps,
        y=subtraction,
        mode='lines',
        name='Subtraction',
        line=dict(color='red', width=1),
        hovertemplate='<b>Subtraction</b><br>Time: %{x}<br>Value: %{y:.2f}°C<extra></extra>'
    ),
    row=3, col=1
)

# Last two weeks
fig.add_trace(
    go.Scatter(
        x=timestamps[mask_last_two_weeks],
        y=subtraction[mask_last_two_weeks],
        mode='lines',
        name='Subtraction',
        line=dict(color='red', width=1),
        showlegend=False,
        hovertemplate='<b>Subtraction</b><br>Time: %{x}<br>Value: %{y:.2f}°C<extra></extra>'
    ),
    row=3, col=2
)

# Update x-axis labels
fig.update_xaxes(title_text="Time", row=3, col=1)
fig.update_xaxes(title_text="Time", row=3, col=2)

# Update y-axis labels
fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
fig.update_yaxes(title_text="Temperature (°C)", row=1, col=2)
fig.update_yaxes(title_text="Temperature (°C)", row=2, col=1)
fig.update_yaxes(title_text="Temperature (°C)", row=2, col=2)
fig.update_yaxes(title_text="Temperature Difference (°C)", row=3, col=1)
fig.update_yaxes(title_text="Temperature Difference (°C)", row=3, col=2)

# Update layout
fig.update_layout(
    height=1200,
    title_text="Temperature Comparison: Panel vs Open-Meteo",
    title_x=0.5,
    hovermode='x unified'
)

# Show plot
fig.show()